In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import BallTree
from tqdm import tqdm
from fuzzywuzzy import fuzz
import warnings
import os

warnings.filterwarnings('ignore')

#### Trying to map the admin and google data first on coordinates

In [ ]:
#25112025 - increaced distance thresholds for confidence levels to 500 meters 
# Create output directory
output_dir = '/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates_24112025/relevant_gmaps_data'
os.makedirs(output_dir, exist_ok=True)


### Functions

In [ ]:
def normalize_dtypes(df):
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            # Convert pandas 'string' dtype → object
            df[col] = df[col].astype(object)

        elif pd.api.types.is_float_dtype(df[col]):
            # Convert nullable Float64 → float64
            if str(df[col].dtype) == "Float64":
                df[col] = df[col].astype("float64")

        elif pd.api.types.is_integer_dtype(df[col]):
            # Convert nullable Int64 → int64
            if str(df[col].dtype) == "Int64":
                df[col] = df[col].astype("float64").astype("int64", errors="ignore")

    return df

In [ ]:
class AllMatchesAdminAnalyzer:
    def __init__(self, threshold_meters=500):
            self.threshold_meters = threshold_meters
            self.all_matches = None
            self.address_matches = None
            self.admin_match_stats = None
            self.df_admin_clean = None
            self.df_gmaps_clean = None
            
            # Confidence levels for admin matching analysis
            self.confidence_levels = {
                'very_high': (0, 10, '0-10m: Very high confidence'),
                'high': (10, 50, '10-50m: High confidence'), 
                'medium': (50, 100, '50-100m: Medium confidence'),
                'low': (100, 500, '100-500m: Low confidence'),
                'address_match': (None, None, 'Address-based match'),
                'no_match': (500, float('inf'), 'No match found')
            }
        
    def haversine_distance(self, lat1, lon1, lat2, lon2):
        """Calculate Haversine distance between two points in meters"""
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        return c * 6371000
    
    def find_all_admin_matches_within_radius(self, df_admin, df_gmaps, lat_col_admin='latitude', lon_col_admin='longitude',
                                           lat_col_gmaps='latitude', lon_col_gmaps='longitude'):
        """
        Find ALL Google Maps matches within 100m radius for each admin point
        Returns multiple matches per admin point with different accuracies
        """
        print(f"Finding ALL matches within {self.threshold_meters}m radius for {len(df_admin):,} admin points...")
        print("Note: Each admin point can have multiple Google Maps matches!")
        
        # Store original dataframes for later use
        self.df_admin_original = df_admin.copy()
        self.df_gmaps_original = df_gmaps.copy()
        
        # Convert coordinates to numeric and handle any conversion errors
        print("Converting coordinates to numeric types...")
        
        # Convert admin coordinates to numeric, coercing errors to NaN
        df_admin[lat_col_admin] = pd.to_numeric(df_admin[lat_col_admin], errors='coerce')
        df_admin[lon_col_admin] = pd.to_numeric(df_admin[lon_col_admin], errors='coerce')
        
        # Convert gmaps coordinates to numeric, coercing errors to NaN  
        df_gmaps[lat_col_gmaps] = pd.to_numeric(df_gmaps[lat_col_gmaps], errors='coerce')
        df_gmaps[lon_col_gmaps] = pd.to_numeric(df_gmaps[lon_col_gmaps], errors='coerce')
        
        # Remove rows with invalid coordinates (NaN values after conversion)
        admin_initial_count = len(df_admin)
        gmaps_initial_count = len(df_gmaps)
        
        df_admin_clean = df_admin.dropna(subset=[lat_col_admin, lon_col_admin]).copy()
        df_gmaps_clean = df_gmaps.dropna(subset=[lat_col_gmaps, lon_col_gmaps]).copy()
        
        # Store cleaned dataframes for export
        self.df_admin_clean = df_admin_clean
        self.df_gmaps_clean = df_gmaps_clean
        
        admin_final_count = len(df_admin_clean)
        gmaps_final_count = len(df_gmaps_clean)
        
        print(f"Admin points after cleaning: {admin_final_count:,} (removed {admin_initial_count - admin_final_count} invalid rows)")
        print(f"GMaps points after cleaning: {gmaps_final_count:,} (removed {gmaps_initial_count - gmaps_final_count} invalid rows)")
        
        # Prepare coordinates - NOW THEY ARE NUMERIC
        coords_gmaps = np.radians(df_gmaps_clean[[lat_col_gmaps, lon_col_gmaps]].values)
        coords_admin = np.radians(df_admin_clean[[lat_col_admin, lon_col_admin]].values)
        
        # Build BallTree for efficient radius search
        tree = BallTree(coords_gmaps, metric='haversine')
        radius_rad = self.threshold_meters / 6371000  # Convert meters to radians
        
        # Find ALL points within radius for each admin point
        print("Searching for all matches within radius...")
        indices_list = tree.query_radius(coords_admin, r=radius_rad)
        
        # Create results for ALL matches (multiple matches per admin point possible)
        results = []
        admin_match_counts = []  # Track how many matches per admin point
        
        for i in tqdm(range(len(df_admin_clean)), desc="Processing admin points"):
            admin_idx = i
            gmaps_indices = indices_list[i]
            
            # Get admin coordinates
            lat_admin = np.degrees(coords_admin[i][0])
            lon_admin = np.degrees(coords_admin[i][1])
            
            # Track match count for this admin point
            match_count = 0
            
            if len(gmaps_indices) > 0:
                # This admin point has matches!
                for gmaps_idx in gmaps_indices:
                    # Get gmaps coordinates
                    lat_gmaps = np.degrees(coords_gmaps[gmaps_idx][0])
                    lon_gmaps = np.degrees(coords_gmaps[gmaps_idx][1])
                    
                    # Calculate exact distance
                    distance = self.haversine_distance(lat_admin, lon_admin, lat_gmaps, lon_gmaps)
                    
                    # Determine confidence level
                    if distance <= 10:
                        confidence = 'very_high'
                    elif distance <= 50:
                        confidence = 'high'
                    elif distance <= 100:
                        confidence = 'medium'    
                    else:  # 100 -500m
                        confidence = 'low'
                    
                    results.append({
                        'admin_index': admin_idx,
                        'gmaps_index': gmaps_idx,
                        'lat_admin': lat_admin,
                        'lon_admin': lon_admin,
                        'lat_gmaps': lat_gmaps,
                        'lon_gmaps': lon_gmaps,
                        'distance_meters': distance,
                        'confidence_level': confidence,
                        'match_description': self.confidence_levels[confidence][2],
                        'is_closest_match': False  # We'll mark the closest one later
                    })
                    match_count += 1
            else:
                # No matches found for this admin point
                results.append({
                    'admin_index': admin_idx,
                    'gmaps_index': -1,
                    'lat_admin': lat_admin,
                    'lon_admin': lon_admin,
                    'lat_gmaps': np.nan,
                    'lon_gmaps': np.nan,
                    'distance_meters': np.nan,
                    'confidence_level': 'no_match',
                    'match_description': self.confidence_levels['no_match'][2],
                    'is_closest_match': False
                })
                match_count = 0
            
            admin_match_counts.append(match_count)
        
        self.all_matches = pd.DataFrame(results)
        
        # Mark the closest match for each admin point
        self._mark_closest_matches()
        
        print(f"\nMatch distribution:")
        total_matches = len(self.all_matches[self.all_matches['gmaps_index'] != -1])
        admin_with_matches = len([x for x in admin_match_counts if x > 0])
        admin_with_multiple = len([x for x in admin_match_counts if x > 1])
        
        print(f"Total matches found: {total_matches:,}")
        print(f"Admin points with at least one match: {admin_with_matches:,} ({admin_with_matches/admin_final_count*100:.1f}%)")
        print(f"Admin points with multiple matches: {admin_with_multiple:,} ({admin_with_multiple/admin_final_count*100:.1f}%)")
        print(f"Average matches per admin point: {np.mean(admin_match_counts):.2f}")
        print(f"Maximum matches for one admin point: {np.max(admin_match_counts)}")
        
        return self.all_matches
    
    def _mark_closest_matches(self):
        """Mark the closest match for each admin point"""
        if self.all_matches is None:
            return
        
        # For admin points with matches, find and mark the closest one
        admin_with_matches = self.all_matches[self.all_matches['gmaps_index'] != -1]['admin_index'].unique()
        
        for admin_idx in admin_with_matches:
            admin_matches = self.all_matches[
                (self.all_matches['admin_index'] == admin_idx) & 
                (self.all_matches['gmaps_index'] != -1)
            ]
            
            if len(admin_matches) > 0:
                closest_idx = admin_matches['distance_meters'].idxmin()
                self.all_matches.loc[closest_idx, 'is_closest_match'] = True
    
    def perform_address_matching(self, admin_address_cols, gmaps_address_cols, 
                               similarity_threshold=0.7, method='fuzzy'):
        """
        Perform address matching on spatially unmatched points
        
        Args:
            admin_address_cols: List of column names in admin data that contain address info
            gmaps_address_cols: List of column names in gmaps data that contain address info  
            similarity_threshold: Minimum similarity score (0-1) to consider a match
            method: 'fuzzy' for fuzzy string matching or 'exact' for exact matching
        """
        print("\n=== PERFORMING ADDRESS MATCHING FOR UNMATCHED POINTS ===")
        
        # Get spatially unmatched admin points
        unmatched_admin = self.create_complete_unmatched_admin_dataset()
        if unmatched_admin is None or len(unmatched_admin) == 0:
            print("No unmatched admin points for address matching")
            return None
        
        # Get unmatched Google Maps points
        unmatched_gmaps = self.create_unmatched_gmaps_dataset()
        if unmatched_gmaps is None or len(unmatched_gmaps) == 0:
            print("No unmatched Google Maps points for address matching")
            return None
        
        print(f"Attempting address matching on {len(unmatched_admin):,} unmatched admin points and {len(unmatched_gmaps):,} unmatched Google Maps points")
        
        # Prepare address strings
        print("Preparing address strings...")
        unmatched_admin['address_string'] = self._create_address_string(unmatched_admin, admin_address_cols)
        unmatched_gmaps['address_string'] = self._create_address_string(unmatched_gmaps, gmaps_address_cols)
        
        # Remove rows with empty address strings
        unmatched_admin = unmatched_admin[unmatched_admin['address_string'].str.len() > 0].copy()
        unmatched_gmaps = unmatched_gmaps[unmatched_gmaps['address_string'].str.len() > 0].copy()
        
        print(f"After address cleaning: {len(unmatched_admin):,} admin points, {len(unmatched_gmaps):,} Google Maps points")
        
        # Perform matching
        address_matches = []
        
        if method == 'fuzzy':
            address_matches = self._fuzzy_address_match(unmatched_admin, unmatched_gmaps, similarity_threshold)
        elif method == 'exact':
            address_matches = self._exact_address_match(unmatched_admin, unmatched_gmaps)
        else:
            print(f"Unknown method: {method}. Using fuzzy matching.")
            address_matches = self._fuzzy_address_match(unmatched_admin, unmatched_gmaps, similarity_threshold)
        
        print(f"Found {len(address_matches):,} address-based matches")
        
        # Create address matches DataFrame
        if address_matches:
            self.address_matches = pd.DataFrame(address_matches)
            
            # Calculate distances for address matches
            self._calculate_address_match_distances()
            
            return self.address_matches
        else:
            print("No address matches found")
            return None
    
    def _create_address_string(self, df, address_cols):
        """Create a standardized address string from multiple columns"""
        def combine_address(row):
            address_parts = []
            for col in address_cols:
                if col in row and pd.notna(row[col]) and str(row[col]).strip():
                    address_parts.append(str(row[col]).strip())
            return ' '.join(address_parts).lower().strip()
        
        return df.apply(combine_address, axis=1)
    
    def _fuzzy_address_match(self, unmatched_admin, unmatched_gmaps, threshold):
        """Perform fuzzy string matching on addresses"""
        from fuzzywuzzy import fuzz
        import re
        
        print("Performing fuzzy address matching...")
        
        matches = []
        admin_addresses = unmatched_admin['address_string'].tolist()
        gmaps_addresses = unmatched_gmaps['address_string'].tolist()
        
        # Clean addresses for better matching
        def clean_address(addr):
            # Remove extra spaces, special characters, standardize common terms
            addr = re.sub(r'[^\w\s]', ' ', addr)
            addr = re.sub(r'\s+', ' ', addr)
            # Standardize common address terms
            replacements = {
                'street': 'st', 'avenue': 'ave', 'road': 'rd', 
                'boulevard': 'blvd', 'drive': 'dr', 'lane': 'ln'
            }
            for full, abbrev in replacements.items():
                addr = re.sub(r'\b' + full + r'\b', abbrev, addr)
            return addr.strip()
        
        admin_cleaned = [clean_address(addr) for addr in admin_addresses]
        gmaps_cleaned = [clean_address(addr) for addr in gmaps_addresses]
        
        # Create a set of already matched gmaps indices to avoid double matching
        matched_gmaps_indices = set()
        
        for i, (admin_idx, admin_addr, admin_clean) in enumerate(zip(
            unmatched_admin.index, admin_addresses, admin_cleaned)):
            
            if len(admin_clean) < 5:  # Skip very short addresses
                continue
                
            best_score = 0
            best_gmaps_idx = None
            best_gmaps_addr = None
            
            for j, (gmaps_idx, gmaps_addr, gmaps_clean) in enumerate(zip(
                unmatched_gmaps.index, gmaps_addresses, gmaps_cleaned)):
                
                if gmaps_idx in matched_gmaps_indices:
                    continue  # This gmaps point already matched to another admin
                    
                if len(gmaps_clean) < 5:
                    continue
                
                # Use token sort ratio for address matching (handles word order differences)
                score = fuzz.token_sort_ratio(admin_clean, gmaps_clean)
                
                if score > best_score and score >= threshold * 100:
                    best_score = score
                    best_gmaps_idx = gmaps_idx
                    best_gmaps_addr = gmaps_addr
            
            if best_gmaps_idx is not None:
                matches.append({
                    'admin_index': admin_idx,
                    'gmaps_index': best_gmaps_idx,
                    'admin_address': admin_addr,
                    'gmaps_address': best_gmaps_addr,
                    'similarity_score': best_score / 100,  # Convert to 0-1 scale
                    'confidence_level': 'address_match',
                    'match_description': f'Address match (similarity: {best_score:.1f}%)',
                    'is_closest_match': True,
                    'distance_meters': None  # Will calculate later
                })
                matched_gmaps_indices.add(best_gmaps_idx)
        
        return matches
    
    def _exact_address_match(self, unmatched_admin, unmatched_gmaps):
        """Perform exact address matching"""
        print("Performing exact address matching...")
        
        matches = []
        
        # Create normalized address strings (lowercase, stripped)
        admin_normalized = unmatched_admin['address_string'].str.lower().str.strip()
        gmaps_normalized = unmatched_gmaps['address_string'].str.lower().str.strip()
        
        # Find exact matches
        for admin_idx, admin_addr in admin_normalized.items():
            if admin_addr in gmaps_normalized.values:
                gmatches = gmaps_normalized[gmaps_normalized == admin_addr]
                for gmaps_idx in gmatches.index:
                    matches.append({
                        'admin_index': admin_idx,
                        'gmaps_index': gmaps_idx,
                        'admin_address': unmatched_admin.loc[admin_idx, 'address_string'],
                        'gmaps_address': unmatched_gmaps.loc[gmaps_idx, 'address_string'],
                        'similarity_score': 1.0,
                        'confidence_level': 'address_match',
                        'match_description': 'Exact address match',
                        'is_closest_match': True,
                        'distance_meters': None
                    })
                    break  # Take first match only
        
        return matches
    
    def _calculate_address_match_distances(self):
        """Calculate actual distances for address matches"""
        if self.address_matches is None:
            return
        
        print("Calculating distances for address matches...")
        
        distances = []
        for _, match in self.address_matches.iterrows():
            admin_idx = match['admin_index']
            gmaps_idx = match['gmaps_index']
            
            # Get coordinates from cleaned dataframes
            admin_coords = self.df_admin_clean.iloc[admin_idx]
            gmaps_coords = self.df_gmaps_clean.iloc[gmaps_idx]
            
            distance = self.haversine_distance(
                admin_coords['latitude'], admin_coords['longitude'],
                gmaps_coords['latitude'], gmaps_coords['longitude']
            )
            distances.append(distance)
        
        self.address_matches['distance_meters'] = distances
        
        # Update confidence based on actual distance
        def update_confidence(row):
            distance = row['distance_meters']
            if distance <= 10:
                return 'very_high'
            elif distance <= 50:
                return 'high'
            elif distance <= 100:
                return 'medium'
            elif distance <= 500:
                return 'low'
            else:
                return 'address_match'  # Keep as address match even if far
            
        self.address_matches['confidence_level'] = self.address_matches.apply(update_confidence, axis=1)
       
    def create_complete_all_matches_dataset(self):
        """Create a complete dataset with ALL columns from both admin and gmaps for ALL matches"""
        if self.all_matches is None or self.df_admin_clean is None or self.df_gmaps_clean is None:
            print("No matches or data available")
            return None
        
        # Get only the actual matches (exclude no-match rows)
        actual_matches = self.all_matches[self.all_matches['gmaps_index'] != -1].copy()
        
        if len(actual_matches) == 0:
            print("No matches found to create complete dataset")
            return None
        
        print(f"Creating complete dataset for {len(actual_matches):,} total matches...")
        
        # Reset indices to ensure proper merging
        df_admin_reset = self.df_admin_clean.reset_index(drop=True)
        df_gmaps_reset = self.df_gmaps_clean.reset_index(drop=True)
        
        # Merge with admin data
        complete_matches = actual_matches.merge(
            df_admin_reset, 
            left_on='admin_index', 
            right_index=True,
            suffixes=('', '_admin_original')
        )
        
        # Merge with gmaps data
        complete_matches = complete_matches.merge(
            df_gmaps_reset, 
            left_on='gmaps_index', 
            right_index=True,
            suffixes=('_admin', '_gmaps')
        )
        
        # Reorder columns to put matching info first
        match_info_cols = ['admin_index', 'gmaps_index', 'distance_meters', 'confidence_level', 
                          'match_description', 'is_closest_match', 'lat_admin', 'lon_admin', 
                          'lat_gmaps', 'lon_gmaps']
        
        admin_cols = [col for col in complete_matches.columns if col not in match_info_cols and not col.endswith('_gmaps')]
        gmaps_cols = [col for col in complete_matches.columns if col.endswith('_gmaps')]
        
        final_columns = match_info_cols + admin_cols + gmaps_cols
        complete_matches = complete_matches[final_columns]
        
        # Sort by admin index and distance for better readability
        complete_matches = complete_matches.sort_values(['admin_index', 'distance_meters'])
        
        return complete_matches
    
    def create_complete_unmatched_admin_dataset(self):
        """Create a complete dataset with ALL columns from admin for unmatched points"""
        if self.all_matches is None or self.df_admin_clean is None:
            print("No matches or admin data available")
            return None
        
        # Get admin points that have no matches at all
        admin_with_matches = set(self.all_matches[self.all_matches['gmaps_index'] != -1]['admin_index'])
        all_admin_indices = set(range(len(self.df_admin_clean)))
        unmatched_admin_indices = all_admin_indices - admin_with_matches
        
        if len(unmatched_admin_indices) == 0:
            print("No unmatched admin points found")
            return None
        
        print(f"Creating complete unmatched admin dataset for {len(unmatched_admin_indices):,} points...")
        
        # Reset indices to ensure proper merging
        df_admin_reset = self.df_admin_clean.reset_index(drop=True)
        
        # Get unmatched admin points
        unmatched_admin = df_admin_reset.loc[list(unmatched_admin_indices)].copy()
        
        # Add match status information
        unmatched_admin['match_status'] = 'unmatched'
        unmatched_admin['admin_index'] = unmatched_admin.index
        
        return unmatched_admin
    
    def create_unmatched_gmaps_dataset(self):
        """Create a dataset with ALL columns from gmaps for points that weren't matched to any admin"""
        if self.all_matches is None or self.df_gmaps_clean is None:
            print("No matches or gmaps data available")
            return None
        
        # Get all gmaps indices that were matched
        matched_gmaps_indices = set(self.all_matches[self.all_matches['gmaps_index'] != -1]['gmaps_index'])
        
        # Get gmaps points that weren't matched
        unmatched_gmaps_mask = ~self.df_gmaps_clean.index.isin(matched_gmaps_indices)
        unmatched_gmaps = self.df_gmaps_clean[unmatched_gmaps_mask].copy()
        
        print(f"Found {len(unmatched_gmaps):,} unmatched Google Maps points")
        return unmatched_gmaps
    
    def create_complete_address_matches_dataset(self):
        """Create a complete dataset with ALL columns from both admin and gmaps for ADDRESS matches"""
        if self.address_matches is None or self.df_admin_clean is None or self.df_gmaps_clean is None:
            print("No address matches or data available")
            return None
        
        if len(self.address_matches) == 0:
            print("No address matches found to create complete dataset")
            return None
        
        print(f"Creating complete dataset for {len(self.address_matches):,} address matches...")
        
        # Reset indices to ensure proper merging
        df_admin_reset = self.df_admin_clean.reset_index(drop=True)
        df_gmaps_reset = self.df_gmaps_clean.reset_index(drop=True)
        
        # Merge with admin data
        complete_address_matches = self.address_matches.merge(
            df_admin_reset, 
            left_on='admin_index', 
            right_index=True,
            suffixes=('', '_admin_original')
        )
        
        # Merge with gmaps data
        complete_address_matches = complete_address_matches.merge(
            df_gmaps_reset, 
            left_on='gmaps_index', 
            right_index=True,
            suffixes=('_admin', '_gmaps')
        )
        
        # Reorder columns to put matching info first
        match_info_cols = ['admin_index', 'gmaps_index', 'distance_meters', 'confidence_level', 
                        'match_description', 'is_closest_match', 'similarity_score',
                        'admin_address', 'gmaps_address', 'latitude_admin', 'longitude_admin', 
                        'latitude_gmaps', 'longitude_gmaps']
        
        admin_cols = [col for col in complete_address_matches.columns if col not in match_info_cols and not col.endswith('_gmaps')]
        gmaps_cols = [col for col in complete_address_matches.columns if col.endswith('_gmaps')]
        
        final_columns = match_info_cols + admin_cols + gmaps_cols
        complete_address_matches = complete_address_matches[final_columns]
        
        # Sort by admin index for better readability
        complete_address_matches = complete_address_matches.sort_values(['admin_index'])
        
        return complete_address_matches
    
    def create_combined_matches_dataset(self):
        """Combine spatial matches and address matches into one dataset"""
        if self.all_matches is None:
            print("No spatial matches available")
            return None
        
        # Get spatial matches (excluding no-match rows)
        spatial_matches = self.all_matches[self.all_matches['gmaps_index'] != -1].copy()
        
        if self.address_matches is not None and len(self.address_matches) > 0:
            # Add address matches
            combined_matches = pd.concat([spatial_matches, self.address_matches], ignore_index=True)
            print(f"Combined dataset: {len(spatial_matches):,} spatial + {len(self.address_matches):,} address = {len(combined_matches):,} total matches")
        else:
            combined_matches = spatial_matches
            print(f"Combined dataset: {len(spatial_matches):,} spatial matches only")
        
        return combined_matches
    
    def analyze_admin_match_rates(self):
        """Comprehensive analysis of how many admin points found matches"""
        if self.all_matches is None:
            return {}
        
        total_admin = len(self.df_admin_clean) if self.df_admin_clean is not None else 0
        
        # Count unique admin points with matches
        admin_with_matches = set(self.all_matches[self.all_matches['gmaps_index'] != -1]['admin_index'])
        matched_count = len(admin_with_matches)
        unmatched_count = total_admin - matched_count
        match_rate = (matched_count / total_admin) * 100
        
        # Count total number of matches (can be > total_admin since multiple matches per admin)
        total_matches = len(self.all_matches[self.all_matches['gmaps_index'] != -1])
        
        # Count admin points with multiple matches
        match_counts = self.all_matches[self.all_matches['gmaps_index'] != -1].groupby('admin_index').size()
        multiple_matches_count = len(match_counts[match_counts > 1])
        
        # Breakdown by confidence level - FIXED: Include 'low' confidence level
        actual_matches = self.all_matches[self.all_matches['gmaps_index'] != -1]
        confidence_counts = {}
        for level in ['very_high', 'high', 'medium', 'low']:  # Added 'low' here
            count = len(actual_matches[actual_matches['confidence_level'] == level])
            confidence_counts[level] = {
                'count': count,
                'percentage': (count / total_matches) * 100
            }
        
        confidence_counts['no_match'] = {
            'count': unmatched_count,
            'percentage': (unmatched_count / total_admin) * 100
        }
        
        # Distance statistics for all matches
        distance_stats = {
            'mean_distance': actual_matches['distance_meters'].mean(),
            'median_distance': actual_matches['distance_meters'].median(),
            'min_distance': actual_matches['distance_meters'].min(),
            'max_distance': actual_matches['distance_meters'].max(),
            'std_distance': actual_matches['distance_meters'].std()
        }
        
        stats = {
            'total_admin_points': total_admin,
            'admin_points_with_matches': matched_count,
            'admin_points_with_multiple_matches': multiple_matches_count,
            'unmatched_admin_points': unmatched_count,
            'total_matches_found': total_matches,
            'overall_match_rate_percent': match_rate,
            'multiple_match_rate_percent': (multiple_matches_count / total_admin) * 100,
            'avg_matches_per_matched_admin': total_matches / matched_count if matched_count > 0 else 0,
            'confidence_breakdown': confidence_counts,
            'distance_statistics': distance_stats
        }
        
        self.admin_match_stats = stats
        return stats
        
    def analyze_combined_match_rates(self):
        """Analyze match rates including both spatial and address matches"""
        spatial_stats = self.analyze_admin_match_rates()
        
        if self.address_matches is None or len(self.address_matches) == 0:
            print("No address matches to combine")
            return spatial_stats
        
        total_admin = spatial_stats['total_admin_points']
        spatial_matched = spatial_stats['admin_points_with_matches']
        address_matched = len(set(self.address_matches['admin_index']))
        
        # Calculate combined stats (avoid double counting)
        combined_matched = spatial_matched + address_matched
        combined_match_rate = (combined_matched / total_admin) * 100
        
        # Count total matches (spatial + address)
        total_spatial_matches = spatial_stats['total_matches_found']
        total_address_matches = len(self.address_matches)
        total_combined_matches = total_spatial_matches + total_address_matches
        
        combined_stats = {
            'total_admin_points': total_admin,
            'spatial_matches_only': spatial_matched,
            'address_matches_only': address_matched,
            'combined_matches_total': combined_matched,
            'spatial_match_rate': spatial_stats['overall_match_rate_percent'],
            'address_match_increase': address_matched,
            'combined_match_rate': combined_match_rate,
            'improvement_from_address': combined_match_rate - spatial_stats['overall_match_rate_percent'],
            'total_spatial_matches': total_spatial_matches,
            'total_address_matches': total_address_matches,
            'total_combined_matches': total_combined_matches
        }
        
        print(f"\n=== COMBINED MATCHING RESULTS ===")
        print(f"Spatial matches only: {spatial_matched:,} ({spatial_stats['overall_match_rate_percent']:.1f}%)")
        print(f"Additional address matches: {address_matched:,}")
        print(f"Combined total: {combined_matched:,} ({combined_match_rate:.1f}%)")
        print(f"Improvement from address matching: +{address_matched:,} points (+{combined_stats['improvement_from_address']:.1f}%)")
        
        return combined_stats
    
    def export_complete_all_matches_results(self):
        """Export COMPLETE results with all columns from both datasets for ALL matches"""
        print(f"\nExporting COMPLETE ALL-MATCHES results to {output_dir}")
        
        # 1. Complete ALL matches dataset with ALL columns from both admin and gmaps
        complete_all_matches = self.create_complete_all_matches_dataset()
        if complete_all_matches is not None:
            complete_all_matches.to_csv(f'{output_dir}complete_ALL_matches_all_columns.csv', index=False)
            print(f"✓ complete_ALL_matches_all_columns.csv: {len(complete_all_matches):,} TOTAL matches with ALL columns from both datasets")
            print(f"  - Includes multiple matches per admin point with different accuracies")
            print(f"  - Marked with 'is_closest_match' column to identify the closest match")
        
        # 2. Closest matches only (for comparison)
        if complete_all_matches is not None:
            closest_matches = complete_all_matches[complete_all_matches['is_closest_match'] == True]
            closest_matches.to_csv(f'{output_dir}closest_matches_only_all_columns.csv', index=False)
            print(f"✓ closest_matches_only_all_columns.csv: {len(closest_matches):,} closest matches only")
        
        # 3. Complete unmatched admin points with ALL admin columns
        complete_unmatched_admin = self.create_complete_unmatched_admin_dataset()
        if complete_unmatched_admin is not None:
            complete_unmatched_admin.to_csv(f'{output_dir}complete_unmatched_admin_all_columns.csv', index=False)
            print(f"✓ complete_unmatched_admin_all_columns.csv: {len(complete_unmatched_admin):,} unmatched admin points with ALL admin columns")
        
        # 4. Unmatched Google Maps points (points that didn't match any admin)
        unmatched_gmaps = self.create_unmatched_gmaps_dataset()
        if unmatched_gmaps is not None:
            unmatched_gmaps.to_csv(f'{output_dir}unmatched_gmaps_points_all_columns.csv', index=False)
            print(f"✓ unmatched_gmaps_points_all_columns.csv: {len(unmatched_gmaps):,} Google Maps points that didn't match any admin")
        
        # 5. Basic all-matches results (for reference)
        self.all_matches.to_csv(f'{output_dir}basic_ALL_matching_results.csv', index=False)
        print(f"✓ basic_ALL_matching_results.csv: {len(self.all_matches):,} basic matching results (all matches)")
        
        # 6. Summary statistics
        summary_df = pd.DataFrame([self.admin_match_stats])
        summary_df.to_csv(f'{output_dir}admin_matching_summary_ALL_matches.csv', index=False)
        print(f"✓ admin_matching_summary_ALL_matches.csv: Comprehensive matching statistics for ALL matches")
        
        return True

    def export_complete_combined_results(self):
        """Export results including both spatial and address matches"""
        print(f"\nExporting COMBINED results to {output_dir}")
        
        try:
            # 1. Export spatial matches (existing functionality)
            self.export_complete_all_matches_results()
            
            # 2. Export address matches - DO THIS FIRST to ensure they're saved
            address_exported = False
            if self.address_matches is not None and len(self.address_matches) > 0:
                try:
                    complete_address_matches = self.create_complete_address_matches_dataset()
                    if complete_address_matches is not None:
                        complete_address_matches.to_csv(f'{output_dir}complete_ADDRESS_matches_all_columns.csv', index=False)
                        print(f"✓ complete_ADDRESS_matches_all_columns.csv: {len(complete_address_matches):,} address-based matches")
                        address_exported = True
                except Exception as e:
                    print(f"⚠ Error creating address matches dataset: {e}")
                    # Try to save the basic address matches as fallback
                    try:
                        self.address_matches.to_csv(f'{output_dir}basic_ADDRESS_matches.csv', index=False)
                        print(f"✓ basic_ADDRESS_matches.csv: {len(self.address_matches):,} address matches (basic format)")
                        address_exported = True
                    except Exception as e2:
                        print(f"❌ Failed to save even basic address matches: {e2}")
            
            # 3. Try to export combined matches (spatial + address)
            combined_matches = self.create_combined_matches_dataset()
            if combined_matches is not None:
                try:
                    complete_combined_matches = self._create_complete_combined_dataset(combined_matches)
                    if complete_combined_matches is not None:
                        complete_combined_matches.to_csv(f'{output_dir}complete_COMBINED_matches_all_columns.csv', index=False)
                        print(f"✓ complete_COMBINED_matches_all_columns.csv: {len(complete_combined_matches):,} total matches (spatial + address)")
                    else:
                        print("⚠ Could not create combined dataset, but individual files are available")
                except Exception as e:
                    print(f"⚠ Error creating combined dataset: {e}")
                    print("⚠ Individual match files are still available")
                
                # Export combined statistics regardless
                try:
                    combined_stats = self.analyze_combined_match_rates()
                    if combined_stats:
                        summary_df = pd.DataFrame([combined_stats])
                        summary_df.to_csv(f'{output_dir}combined_matching_summary.csv', index=False)
                        print(f"✓ combined_matching_summary.csv: Combined statistics (spatial + address)")
                except Exception as e:
                    print(f"⚠ Error creating combined statistics: {e}")
            
            # 4. Export updated unmatched datasets (after address matching)
            if address_exported and self.address_matches is not None and len(self.address_matches) > 0:
                try:
                    # Get admin points that are still unmatched after address matching
                    admin_with_spatial_matches = set(self.all_matches[self.all_matches['gmaps_index'] != -1]['admin_index'])
                    admin_with_address_matches = set(self.address_matches['admin_index'])
                    all_matched_admin = admin_with_spatial_matches.union(admin_with_address_matches)
                    
                    # Get still unmatched admin points
                    all_admin_indices = set(range(len(self.df_admin_clean)))
                    still_unmatched_admin_indices = all_admin_indices - all_matched_admin
                    
                    if len(still_unmatched_admin_indices) > 0:
                        df_admin_reset = self.df_admin_clean.reset_index(drop=True)
                        still_unmatched_admin = df_admin_reset.loc[list(still_unmatched_admin_indices)].copy()
                        still_unmatched_admin['match_status'] = 'unmatched_after_address_matching'
                        still_unmatched_admin.to_csv(f'{output_dir}still_unmatched_admin_after_address_matching.csv', index=False)
                        print(f"✓ still_unmatched_admin_after_address_matching.csv: {len(still_unmatched_admin):,} admin points still unmatched after address matching")
                except Exception as e:
                    print(f"⚠ Error creating unmatched dataset: {e}")
        
        except Exception as e:
            print(f"❌ Major error during export: {e}")
            print("Trying emergency export of individual components...")
            
            # Emergency fallback: export whatever we can
            try:
                # Export basic spatial matches
                if self.all_matches is not None:
                    self.all_matches.to_csv(f'{output_dir}emergency_basic_matches.csv', index=False)
                    print(f"✓ emergency_basic_matches.csv: Basic match data saved")
            except:
                print("❌ Could not save emergency spatial matches")
            
            try:
                # Export basic address matches
                if self.address_matches is not None:
                    self.address_matches.to_csv(f'{output_dir}emergency_address_matches.csv', index=False)
                    print(f"✓ emergency_address_matches.csv: Basic address matches saved")
            except:
                print("❌ Could not save emergency address matches")
        
        return True
    
    def _create_complete_combined_dataset(self, combined_matches):
        """Helper function to create complete dataset for combined matches"""
        if combined_matches is None or len(combined_matches) == 0:
            return None
        
        print("Creating combined dataset...")
        
        # Instead of trying to merge separately, let's create a simpler approach
        # Get complete spatial matches
        complete_spatial = self.create_complete_all_matches_dataset()
        
        # Get complete address matches
        complete_address = self.create_complete_address_matches_dataset()
        
        # If we have both, combine them carefully
        if complete_spatial is not None and complete_address is not None:
            print(f"Combining {len(complete_spatial):,} spatial + {len(complete_address):,} address matches")
            
            # Find common columns between the two datasets
            spatial_cols = set(complete_spatial.columns)
            address_cols = set(complete_address.columns)
            common_cols = list(spatial_cols & address_cols)
            
            print(f"Found {len(common_cols)} common columns for merging")
            
            # If there are enough common columns, combine them
            if len(common_cols) >= 5:  # arbitrary minimum
                # Ensure we have the key columns
                key_cols = ['admin_index', 'gmaps_index', 'distance_meters', 'confidence_level']
                for col in key_cols:
                    if col not in common_cols:
                        common_cols.append(col)
                
                # Take only the common columns from each dataset
                spatial_common = complete_spatial[common_cols].copy()
                address_common = complete_address[common_cols].copy()
                
                # Add a match_type column to distinguish
                spatial_common['match_type'] = 'spatial'
                address_common['match_type'] = 'address'
                
                # Combine
                complete_combined = pd.concat([spatial_common, address_common], ignore_index=True)
                
                # Sort
                complete_combined = complete_combined.sort_values(['admin_index', 'match_type'])
                
                return complete_combined
            else:
                print("Warning: Not enough common columns to combine datasets")
                return complete_spatial  # Return spatial as fallback
        elif complete_spatial is not None:
            print(f"Returning {len(complete_spatial):,} spatial matches only")
            complete_spatial['match_type'] = 'spatial'
            return complete_spatial
        elif complete_address is not None:
            print(f"Returning {len(complete_address):,} address matches only")
            complete_address['match_type'] = 'address'
            return complete_address
        else:
            print("No matches to combine")
            return None

In [ ]:
def run_complete_matching_analysis(df_admin, df_gmaps, admin_address_cols=None, gmaps_address_cols=None, return_stats=True):
    """
    Run complete matching analysis with both spatial and optional address matching
    
    Args:
        df_admin: DataFrame containing administrative points
        df_gmaps: DataFrame containing Google Maps points
        admin_address_cols: List of admin address columns for address matching (optional)
        gmaps_address_cols: List of gmaps address columns for address matching (optional)
        return_stats: Whether to return statistics along with analyzer (default: True)
    """
    print("=== COMPLETE MATCHING ANALYSIS ===")
    
    # Show dataset info
    print(f"Admin dataset: {len(df_admin):,} points")
    print(f"Google Maps dataset: {len(df_gmaps):,} points")
    
    # Show column information
    print(f"\nAdmin dataset columns: {list(df_admin.columns)}")
    print(f"GMaps dataset columns: {list(df_gmaps.columns)}")
    
    # Initialize analyzer
    analyzer = AllMatchesAdminAnalyzer(threshold_meters=500)
    
    # Step 1: Spatial matching
    print("\n=== STEP 1: SPATIAL MATCHING ===")
    all_matches = analyzer.find_all_admin_matches_within_radius(df_admin, df_gmaps)
    spatial_stats = analyzer.analyze_admin_match_rates()
    
    # Print spatial results
    print(f"\n=== SPATIAL MATCHING RESULTS ===")
    print(f"Total admin points: {spatial_stats['total_admin_points']:,}")
    print(f"Admin points with at least one match: {spatial_stats['admin_points_with_matches']:,} ({spatial_stats['overall_match_rate_percent']:.1f}%)")
    print(f"Admin points with MULTIPLE matches: {spatial_stats['admin_points_with_multiple_matches']:,} ({spatial_stats['multiple_match_rate_percent']:.1f}%)")
    print(f"Unmatched admin points: {spatial_stats['unmatched_admin_points']:,} ({100-spatial_stats['overall_match_rate_percent']:.1f}%)")
    print(f"Total matches found: {spatial_stats['total_matches_found']:,}")
    print(f"Average matches per matched admin point: {spatial_stats['avg_matches_per_matched_admin']:.2f}")
    
    print(f"\nMatch Quality Breakdown (across all matches):")
    for level in ['very_high', 'high', 'medium', 'low', 'no_match']:
        level_data = spatial_stats['confidence_breakdown'][level]
        description = analyzer.confidence_levels.get(level, ('', '', 'No match'))[2]
        print(f"  {description:<25}: {level_data['count']:>6,} ({level_data['percentage']:5.1f}%)")
    
    # Show sample of multiple matches
    print(f"\n=== SAMPLE OF ADMIN POINTS WITH MULTIPLE MATCHES ===")
    complete_matches = analyzer.create_complete_all_matches_dataset()
    if complete_matches is not None and len(complete_matches) > 0:
        admin_match_counts = complete_matches['admin_index'].value_counts()
        admin_with_multiple = admin_match_counts[admin_match_counts > 1].index
        
        if len(admin_with_multiple) > 0:
            sample_admin = admin_with_multiple[0]
            sample_matches = complete_matches[complete_matches['admin_index'] == sample_admin]
            
            print(f"Admin point {sample_admin} has {len(sample_matches)} matches:")
            sample_cols = [col for col in sample_matches.columns if any(x in col for x in ['distance', 'confidence', 'is_closest', 'name', 'id'])]
            sample_cols = sample_cols[:6]
            print(sample_matches[sample_cols].to_string(index=False))
        else:
            print("No admin points with multiple matches found")
    
    # Step 2: Address matching (if address columns provided)
    if admin_address_cols and gmaps_address_cols:
        print("\n=== STEP 2: ADDRESS MATCHING ===")
        address_matches = analyzer.perform_address_matching(
            admin_address_cols=admin_address_cols,
            gmaps_address_cols=gmaps_address_cols,
            similarity_threshold=0.7,
            method='fuzzy'
        )
        
        # Step 3: Combined analysis
        print("\n=== STEP 3: COMBINED ANALYSIS ===")
        combined_stats = analyzer.analyze_combined_match_rates()
        
        # Export combined results
        print("\n=== STEP 4: EXPORTING COMBINED RESULTS ===")
        analyzer.export_complete_combined_results()
        
        final_stats = combined_stats
    else:
        print("\nSkipping address matching (no address columns provided)")
        analyzer.export_complete_all_matches_results()
        final_stats = spatial_stats
    
    if return_stats:
        return analyzer, final_stats
    else:
        return analyzer
    

In [ ]:
def correct_coordinates_in_place(df, lat_col='latitude', lon_col='longitude'):
    """
    Correct the coordinate columns in place by replacing first comma with dot and removing others
    """
    # Store original counts for reporting
    original_lat_problems = df[lat_col].astype(str).str.contains(',').sum()
    original_lon_problems = df[lon_col].astype(str).str.contains(',').sum()
    
    # Function to apply the transformation
    def transform_coordinate(value):
        if isinstance(value, str) and ',' in value:
            parts = value.split(',')
            return parts[0] + '.' + ''.join(parts[1:])
        return value
    
    # Apply transformations
    df[lat_col] = df[lat_col].apply(transform_coordinate)
    df[lon_col] = df[lon_col].apply(transform_coordinate)
    
    # Convert to float64
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # Report what was fixed
    print(f"Fixed {original_lat_problems} latitude values and {original_lon_problems} longitude values")
    print(f"Data types after correction:")
    print(f"  {lat_col}: {df[lat_col].dtype}")
    print(f"  {lon_col}: {df[lon_col].dtype}")
    
    return df

### Read Datasets

In [ ]:
### original pools from the administrative  website
admin_pools = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/groundtruth_pools/baederleben_pools_16112025.csv')
admin_pools['address'] = admin_pools['street'] + ', ' + admin_pools['postal_code'].astype(str) + ' ' + admin_pools['city'] + ', Germany'
admin_pools.info()

In [ ]:
duplicate_coords = admin_pools[admin_pools.duplicated(subset=['latitude', 'longitude'], keep=False)]
duplicate_coords = duplicate_coords.sort_values(by=['latitude', 'longitude'])
duplicate_coords

In [ ]:
## see duplicate addressses but kept all entries
duplicate_addresses = admin_pools[admin_pools.duplicated(subset=['address'], keep=False)]
duplicate_addresses = duplicate_addresses.sort_values(by='address')
duplicate_addresses

In [ ]:
admin_pools_cleaned =admin_pools.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
### clean  coordinates
admin_pools_cleaned = correct_coordinates_in_place(admin_pools_cleaned, lat_col='latitude', lon_col='longitude')
# 
admin_pools_cleaned = admin_pools_cleaned.drop_duplicates(subset=['latitude', 'longitude']).reset_index(drop=True)
admin_pools_cleaned.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/groundtruth_pools/baederleben_pools_16112025_cleaned.csv', index=False)
# admin_pools_cleaned = admin_pools_cleaned.drop_duplicates(subset=['address']).reset_index(drop=True) 
admin_pools_cleaned.info()   

In [ ]:
### all google pools collected and merged
gmaps_pools = gpd.read_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/merged_all_gmaps_pids_24112025.gpkg')
gmaps_pools.info()

In [ ]:
relevant_place_types = [
    'aquatic center','athletic club','amusement park','beach','beach volleyball area','baby swimming school',
    'campground','club','fitness center','gym','hotel','indoor swimming pool','lake','lake shore swimming area',
    'lido','medical spa','outdoor bath','outdoor swimming pool','public bath','public bathroom',
    'public wheelchair-accessible bathroom','public sauna','public beach','public swimming pool',
    'recreation center','sauna','sauna club','spa','spa and health club','sports complex','sports school',
    'swim club','swimming basin','swimming facility','swimming lake','swimming pool','swimming school',
    'thermal baths','tourist attraction','water','water park','wellness center','wellness hotel'
]


In [ ]:
gmaps_pools_rev = gmaps_pools[
    (gmaps_pools['primary_place_type'].str.lower().isin(relevant_place_types)) |
    (gmaps_pools['place_subtype'].apply(
        lambda x: any(item.lower() in relevant_place_types for item in x) 
        if isinstance(x, list) else False
    ))
].reset_index(drop=True)
gmaps_pools_rev.info()

### matching google pools and admin pools 

In [ ]:
# # Option 1: Spatial-only matching (returns analyzer + stats)
# analyzer, stats = run_complete_matching_analysis(admin_pools_cleaned, gmaps_pools)

# Option 2: Complete matching with addresses (returns analyzer + combined_stats)
analyzer, combined_stats = run_complete_matching_analysis(
    admin_pools_cleaned, 
    gmaps_pools_rev,
    admin_address_cols=['address'],
    gmaps_address_cols=['address']
)

# # Option 3: Just get analyzer without stats
# analyzer = run_complete_matching_analysis(
#     admin_pools_cleaned, 
#     gmaps_pools,
#     admin_address_cols=['name', 'address', 'city'],
#     gmaps_address_cols=['name', 'formatted_address'],
#     return_stats=False
# )

# Check too see what is missing and quantify

### matched on selected google data

In [ ]:
final_pools = pd.read_csv('/home/ethel/Desktop/WEBCONF_25/datasets/final_swimmingpool_list_corrected_23112025.csv')
final_pools.info()

In [ ]:
#check against matched pools
all_initial_matches =  pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates/complete_ALL_matches_all_columns.csv')
all_initial_matches.info()

### matched on selected google data

In [ ]:
all_matches = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates_24112025/relevant_gmaps_data/relevant_gmaps_datacomplete_ALL_matches_all_columns.csv')
all_matches.info()

In [ ]:
matched_addresses = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates_24112025/relevant_gmaps_data/relevant_gmaps_datacomplete_ADDRESS_matches_all_columns.csv')
matched_addresses.info()

In [ ]:
all_matches = pd.concat([all_matches, matched_addresses], ignore_index=True)
all_matches.info()

In [ ]:
old_fpools = pd.read_csv('/home/ethel/Desktop/WEBCONF_25/datasets/final_swimmingpool_list_corrected_23112025.csv')
old_fpools.info()

In [ ]:
already_in = all_matches[all_matches['place_id'].isin(old_fpools['place_id'])]
# already_in = already_in.drop_duplicates(subset=['place_id']).reset_index(drop=True)
already_in.to_csv('/home/ethel/Desktop/WEBCONF_25/data_cleaning/relevant_gmaps_pids/rev_already_in_final_pools_24112025.csv', index=False)
already_in.info()

In [ ]:
already_in.primary_place_type.value_counts()

In [ ]:
not_in = all_matches[~all_matches['place_id'].isin(old_fpools['place_id'])]
not_in.to_csv('/home/ethel/Desktop/WEBCONF_25/data_cleaning/relevant_gmaps_pids/rev_not_in_final_pools_24112025.csv', index=False)
not_in.info()

In [ ]:
not_in.primary_place_type.value_counts()

In [ ]:
unmatched_admin = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates_24112025/relevant_gmaps_data/relevant_gmaps_datastill_unmatched_admin_after_address_matching.csv')
unmatched_admin.info()

In [ ]:
unmatched_admin.to_csv('/home/ethel/Desktop/WEBCONF_25/data_cleaning/relevant_gmaps_pids/rev_still_unmatched_admin_pools_24112025.csv', index=False )

In [ ]:
unmatched_google = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates_24112025/relevant_gmaps_data/relevant_gmaps_dataunmatched_gmaps_points_all_columns.csv')
unmatched_google.info()

In [ ]:
unmatched_google.to_csv('/home/ethel/Desktop/WEBCONF_25/data_cleaning/relevant_gmaps_pids/rev_unmatched_google_pools_24112025.csv', index=False )

## try for the cleaned dataset

In [ ]:
final_pools = pd.read_csv('/home/ethel/Desktop/WEBCONF_25/datasets/final_swimmingpool_list_corrected_23112025.csv')
final_pools.info()

In [ ]:
final_pools.primary_place_type.value_counts()

In [ ]:
#check against matched pools
all_initial_matches =  pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates/complete_ALL_matches_all_columns.csv')
all_initial_matches.info()


In [ ]:
admin_still_kept = all_initial_matches[all_initial_matches['place_id'].isin(final_pools['place_id'])]
admin_still_kept = admin_still_kept.drop_duplicates(subset=['place_id']).reset_index(drop=True)
admin_still_kept.pool_type.value_counts()

In [ ]:
admin_still_kept.info()

In [ ]:
admin_pools.pool_type.value_counts()

In [ ]:
admin_pools

# Check for the already collected reviews

In [ ]:
matched_pools = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/matching_coordinates/complete_matched_pairs_all_columns.csv")
matched_pools.info()

In [ ]:
#reviews collected from google places
reviews_set_1 = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/clean_datasets/merged_reviews.parquet")
reviews_set_1 = reviews_set_1.drop_duplicates(subset=['review_id']).reset_index(drop=True)
reviews_set_1.info()

In [ ]:
reviews_matched1 = reviews_set_1[reviews_set_1['place_id'].isin(matched_pools['place_id'])]
reviews_matched1.nunique()

In [ ]:
reviews_set_2 = pd.read_csv("/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/all_swim_reviews_germany.csv")
reviews_set_2 = reviews_set_2.drop_duplicates(subset=['review_id']).reset_index(drop=True)
reviews_set_2.info()

In [ ]:
reviews_matched2 = reviews_set_2[reviews_set_2['place_id'].isin(matched_pools['place_id'])]
reviews_matched2.nunique()

In [ ]:
reviews_matched = pd.concat([reviews_matched1, reviews_matched2], axis=0).drop_duplicates(subset=['review_id']).reset_index(drop=True)
reviews_matched.info()

In [ ]:
reviews_matched.nunique()

In [ ]:
import json

def _serialize_for_parquet(val):
	# Keep None/NaN as None, serialize lists/dicts to JSON strings,
	# decode bytes, otherwise cast to str.
	if pd.isna(val):
		return None
	if isinstance(val, (list, dict)):
		try:
			return json.dumps(val, ensure_ascii=False)
		except Exception:
			return str(val)
	if isinstance(val, (bytes, bytearray)):
		try:
			return val.decode('utf-8', errors='replace')
		except Exception:
			return str(val)
	return str(val)

# Serialize problematic object columns to safe string/None values
if 'uploaded_pics' in reviews_matched.columns:
	reviews_matched['uploaded_pics'] = reviews_matched['uploaded_pics'].apply(_serialize_for_parquet)

# If you have other object columns that may contain lists/bytes, handle them similarly:
for col in ['uploaded_pics_links', 'uploaded_pics']:  # add other columns if needed
	if col in reviews_matched.columns:
		reviews_matched[col] = reviews_matched[col].apply(_serialize_for_parquet)

# Now write to parquet
reviews_matched.to_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/reviews_matched_16112025.parquet", index=False)

In [ ]:
reviews_matched.info()

In [ ]:
pids_noreviews = matched_pools[~matched_pools['place_id'].isin(reviews_matched['place_id'])][['place_id', 'unique_identifier']]
pids_noreviews.info()


In [ ]:
pids_noreviews.to_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/more_reviews_json/pids_noreviews_yet_16112025.csv", index=False)

## Complete list of pools to work with now check for reviews

In [ ]:
f_pools = pd.read_csv("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/data/final_swimmingpool_list.csv")
f_pools = f_pools.drop_duplicates(subset=['place_id']).reset_index(drop=True)
f_pools.info()

In [ ]:
f_pools.to_csv("/home/ethel/Desktop/WEBCONF_25/datasets/final_swimmingpool_list_deduplicated.csv", index=False)

In [ ]:
ff_pools = pd.read_parquet("/home/ethel/Desktop/WEBCONF_25/datasets/final_swimmingpool_list.parquet")
ff_pools = ff_pools.drop_duplicates(subset=['place_id']).reset_index(drop=True)
ff_pools.to_parquet("/home/ethel/Desktop/WEBCONF_25/datasets/final_swimmingpool_list_deduplicated.parquet", index=False)    
ff_pools.info()

In [ ]:
collected_reviews = pd.concat([reviews_set_1, reviews_set_2], axis=0).drop_duplicates(subset=['review_id']).reset_index(drop=True)
relevant_reviews = collected_reviews[collected_reviews['place_id'].isin(f_pools['place_id'])].reset_index(drop=True)
relevant_reviews.info()

###  newly collected reviews 21.11.2025

In [ ]:
missed_reviews_21 = pd.read_parquet("/home/ethel/Desktop/WEBCONF_25/clean_datasets/merged_reviews.parquet")
missed_reviews_21 = missed_reviews_21.drop_duplicates(subset=['review_id']).reset_index(drop=True)
missed_reviews_21.info()

In [ ]:
missed_reviews_16 = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/merged_reviews.parquet")
missed_reviews_16 = missed_reviews_16.drop_duplicates(subset=['review_id']).reset_index(drop=True)
missed_reviews_16.info()

In [ ]:
all_missed = pd.concat([missed_reviews_16, missed_reviews_21], axis=0).drop_duplicates(subset=['review_id']).reset_index(drop=True)
all_missed = all_missed[~all_missed['review_id'].isin(relevant_reviews['review_id'])].reset_index(drop=True)
all_missed.info()

In [ ]:
all_missed = all_missed[all_missed['place_id'].isin(f_pools['place_id'])].reset_index(drop=True)
all_missed.info()

In [ ]:
all_reviews_needed = pd.concat([relevant_reviews, all_missed], axis=0).drop_duplicates(subset=['review_id']).reset_index(drop=True)
all_reviews_needed = all_reviews_needed[all_reviews_needed['place_id'].isin(f_pools['place_id'])].reset_index(drop=True)
all_reviews_needed  = all_reviews_needed.drop_duplicates(subset=['review_id']).reset_index(drop=True)
all_reviews_needed.info()

In [ ]:
still_missing = f_pools[~f_pools['place_id'].isin(all_reviews_needed['place_id'])][['place_id', 'unique_identifier','place_name','primary_place_type', 'place_subtype','total_reviews', 'google_url']]
still_missing = still_missing.dropna(subset =['total_reviews'])
still_missing.info()

In [ ]:
still_missing.total_reviews.unique()

## for pois with reviews collected are there still some missing?

In [ ]:
f_pools.total_reviews.sum()

In [ ]:
reviews_incomp = all_reviews_needed.groupby('place_id').size().reset_index(name='num_reviews_collected')
reviews_incomp = reviews_incomp.merge(f_pools[['place_id', 'unique_identifier','total_reviews']], on='place_id', how='left')
reviews_incomp['reviews_missing'] = reviews_incomp['total_reviews'] - reviews_incomp['num_reviews_collected']
reviews_incomp = reviews_incomp[reviews_incomp['reviews_missing'] >5].reset_index(drop=True)
reviews_incomp

In [ ]:
reviews_incomp.total_reviews.sum()

In [ ]:
still_missing.total_reviews.sum()

In [ ]:
rerun_reviews = pd.concat([still_missing[['place_id', 'unique_identifier','total_reviews']], reviews_incomp[['place_id', 'unique_identifier','total_reviews']]], axis=0).drop_duplicates().reset_index(drop=True)
# use division (/) not integer floor-division (//) so ceil yields the correct number of chunks
rerun_reviews['expected_jsons'] = np.ceil(rerun_reviews['total_reviews'] / 10).astype(int)

rerun_reviews.info()

In [ ]:
rerun_reviews.total_reviews.sum()

In [ ]:
rerun_reviews.to_csv("/home/ethel/Desktop/WEBCONF_25/missing_reviews/rerun_missed_reviews.csv", index=False)